[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/Pesquisa-Operacional-III-A/blob/main/03_ArvoreGeradoraMinima.ipynb)

# UNIVERSIDADE FEDERAL FLUMINENSE

**TEP00187 - PESQUISA OPERACIONAL III-A**

**Prof.: Diogo Ferreira de Lima Silva**

Monitores:
- 2023 - Henrique Monteiro Soares da Silva
- 2022 - Rodrigo Celso de Lima Porto

---
# AULA 03 — ÁRVORE GERADORA MÍNIMA
---

## Objetivos da aula

Ao final desta aula você deverá ser capaz de:

1. Reconhecer quando um problema real é um problema de **Árvore Geradora Mínima (AGM)**;
2. Explicar a ideia dos algoritmos de **Kruskal** e de **Prim**;
3. Resolver instâncias de AGM com o NetworkX e interpretar a solução;
4. Visualizar a solução sobre a rede original;
5. Tratar variações do problema: árvore geradora **máxima**, **floresta** geradora mínima e arestas **obrigatórias/proibidas**.

## Roteiro

| Seção | Assunto |
|---|---|
| 1 | Bibliotecas |
| 2 | Revisão: árvores e árvores geradoras |
| 3 | O problema da Árvore Geradora Mínima |
| 4 | Resolvendo com o NetworkX |
| 5 | Uma instância maior: backbone nacional de fibra óptica |
| 6 | Variações do problema |
| 7 | Por que um algoritmo guloso funciona? |
| 8 | Exercícios |

> **Pré-requisito:** Aula 02 (introdução a grafos com NetworkX).

---
## 1. Bibliotecas
---

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

print("networkx:", nx.__version__)

---
## 2. Revisão: árvores e árvores geradoras
---

Uma **árvore** é um grafo **conexo** e **acíclico**. Dessa definição decorrem
propriedades equivalentes, muito úteis na prática:

- uma árvore com $n$ vértices tem **exatamente** $n - 1$ arestas;
- existe **um único** caminho entre qualquer par de vértices;
- **remover** qualquer aresta desconecta a árvore (toda aresta é uma "ponte");
- **acrescentar** qualquer aresta cria exatamente um ciclo.

Uma **floresta** é um grafo acíclico que pode ser desconexo — ou seja, um conjunto
de árvores.

Vamos conferir isso com o NetworkX.

In [ ]:
arvore   = nx.Graph([("a", "b"), ("a", "c"), ("b", "d"), ("b", "e"), ("c", "f")])
ciclo    = nx.Graph([("a", "b"), ("b", "c"), ("c", "a")])
floresta = nx.Graph([("a", "b"), ("c", "d"), ("d", "e")])

for nome, g in [("árvore", arvore), ("ciclo", ciclo), ("floresta", floresta)]:
    print(f"{nome:<9} | conexo: {str(nx.is_connected(g)):<5} | "
          f"acíclico: {str(len(nx.cycle_basis(g)) == 0):<5} | "
          f"|V|={g.number_of_nodes()} |A|={g.number_of_edges()} | "
          f"is_tree: {str(nx.is_tree(g)):<5} | is_forest: {nx.is_forest(g)}")

### 2.1 Árvore geradora

Dado um grafo conexo $G = (V, A)$, uma **árvore geradora** (*spanning tree*) é um
subgrafo $T = (V, A_T)$ tal que:

- $T$ é uma **árvore**;
- $T$ contém **todos** os vértices de $G$ (por isso "geradora");
- $A_T \subseteq A$: usa apenas arestas que já existem em $G$.

Ou seja: é o "esqueleto" mínimo que mantém a rede inteira conectada. Ele terá
sempre $|V| - 1$ arestas, **independentemente** de quais arestas escolhermos.

Em geral, um grafo tem **muitas** árvores geradoras. O NetworkX consegue contá-las
(`nx.number_of_spanning_trees`, via o Teorema de Kirchhoff) e até **enumerá-las**
(`nx.SpanningTreeIterator`).

In [ ]:
P = nx.Graph([(1, 2), (1, 3), (2, 3), (2, 4), (3, 4)])
pos_P = {1: (0, 1), 2: (1, 1), 3: (0, 0), 4: (1, 0)}

print(f"O grafo tem {P.number_of_nodes()} vértices e {P.number_of_edges()} arestas.")
print(f"Toda árvore geradora dele terá {P.number_of_nodes() - 1} arestas.")
print(f"Número total de árvores geradoras: {round(nx.number_of_spanning_trees(P))}")

In [ ]:
arvores = list(nx.SpanningTreeIterator(P))

fig, eixos = plt.subplots(2, 4, figsize=(14, 6))

for eixo, T in zip(eixos.flatten(), arvores):
    nx.draw_networkx_edges(P, pos_P, ax=eixo, edge_color="lightgray", width=1)
    nx.draw_networkx_edges(T, pos_P, ax=eixo, edge_color="crimson", width=3)
    nx.draw_networkx_nodes(P, pos_P, ax=eixo, node_color="#1D3557", node_size=450)
    nx.draw_networkx_labels(P, pos_P, ax=eixo, font_color="white", font_weight="bold")
    eixo.set_title(f"{sorted(T.edges)}", fontsize=8)
    eixo.axis("off")

fig.suptitle(f"As {len(arvores)} árvores geradoras do grafo (em vermelho)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. O problema da Árvore Geradora Mínima
---

Se cada aresta $\{i,j\}$ tem um custo $c_{ij}$, cada árvore geradora tem um custo
total. O problema da **Árvore Geradora Mínima** (*Minimum Spanning Tree*, AGM ou MST)
consiste em encontrar, entre todas as árvores geradoras, a de **menor custo total**:

$$\min_{T \text{ árvore geradora de } G} \; \sum_{\{i,j\} \in A_T} c_{ij}$$

**Onde isso aparece na prática?**

- projetar uma rede de **fibra óptica** ligando um conjunto de localidades com o menor comprimento de cabo;
- decidir quais **linhas de transmissão** de energia construir para atender todas as subestações;
- traçar uma rede de **dutos** (água, gás, esgoto) de custo mínimo;
- definir o **backbone** de uma rede de computadores;
- agrupamento de dados (*clustering*) — o algoritmo *single-linkage* é, essencialmente, uma AGM.

**Cuidado com a interpretação!** A AGM garante que todos os pontos fiquem
conectados pelo menor custo **de construção**. Ela **não** garante o caminho mais
curto entre dois pontos específicos — esse é o problema da Aula 04. São problemas
diferentes, com soluções em geral diferentes.

### 3.1 Os algoritmos clássicos

Os dois algoritmos abaixo são **gulosos** (tomam a decisão localmente mais barata a
cada passo) e, ainda assim, **sempre encontram a solução ótima**.

**Kruskal (1956)** — olha as arestas da mais barata para a mais cara:

```
1. Ordene todas as arestas em ordem crescente de custo.
2. Comece com a solução vazia (cada vértice é uma "ilha").
3. Para cada aresta {u,v} na ordem:
       se u e v estão em ilhas DIFERENTES: aceite (as ilhas se fundem)
       senão:                              rejeite (formaria um ciclo)
4. Pare quando tiver |V| - 1 arestas.
```

**Prim (1957)** — faz a árvore **crescer** a partir de um vértice:

```
1. Escolha um vértice raiz e marque-o como visitado.
2. Repita até visitar todos os vértices:
       entre as arestas que ligam a árvore atual a um vértice AINDA NÃO visitado,
       escolha a mais barata e acrescente-a à árvore.
```

Diferenças práticas:

| | Kruskal | Prim |
|---|---|---|
| Solução parcial | uma **floresta** (vários pedaços que só se juntam no fim) | uma **árvore única** que cresce |
| Complexidade | $O(|A| \log |A|)$ | $O(|A| \log |V|)$ |
| Vai melhor em | grafos **esparsos** | grafos **densos** |

Os dois chegam ao **mesmo custo total**. As arestas escolhidas podem diferir
apenas quando há **empates** de custo.

> **Observação sobre a disciplina:** a AGM é o único problema de otimização em
> redes que veremos **sem** formulá-lo como um Problema de Programação Linear.
> Os algoritmos gulosos resolvem-no de forma tão eficiente que a formulação por
> PL (que exigiria um número exponencial de restrições de eliminação de subciclos)
> não compensa.

Na prática **não implementamos** esses algoritmos: chamamos o NetworkX.

---
## 4. Resolvendo com o NetworkX
---

As funções principais são:

| Função | O que devolve |
|---|---|
| `nx.minimum_spanning_tree(G, weight=..., algorithm=...)` | um **grafo** com a AGM |
| `nx.minimum_spanning_edges(G, ...)` | um **gerador** com as arestas escolhidas |
| `nx.maximum_spanning_tree(G, ...)` | a árvore geradora de custo **máximo** |

O parâmetro `algorithm` aceita `'kruskal'` (padrão) e `'prim'`.

Vamos usar um grafo pequeno, com 6 vértices (`A` a `F`). O custo é gravado no
atributo `weight`, que é o nome que o NetworkX usa por padrão.

In [ ]:
arestas = [("A", "B", 4), ("A", "C", 2), ("B", "C", 1), ("B", "D", 5),
           ("C", "D", 8), ("C", "E", 10), ("D", "E", 2), ("D", "F", 6),
           ("E", "F", 3)]

G = nx.Graph()
G.add_weighted_edges_from(arestas)          # grava no atributo 'weight'

pos = {"A": (0, 1), "B": (1, 2), "C": (1, 0), "D": (2, 1), "E": (3, 0), "F": (3, 2)}

print(f"|V| = {G.number_of_nodes()}  |A| = {G.number_of_edges()}")
print(f"A rede é conexa? {nx.is_connected(G)}  (condição para existir árvore geradora)")
print(f"Qualquer árvore geradora terá {G.number_of_nodes() - 1} arestas.")
print(f"Número de árvores geradoras possíveis: {round(nx.number_of_spanning_trees(G))}")
print(f"Custo total se construíssemos TODAS as arestas: {G.size(weight='weight'):.0f}")

Vamos criar uma função de desenho reaproveitável ao longo da aula. Ela recebe a
lista de arestas que queremos **destacar** — assim conseguimos mostrar a solução
sobre a rede original, mantendo as mesmas posições dos vértices.

In [ ]:
def desenhar_rede(G, pos, ax, titulo, destaque=None, peso="weight",
                  mostrar_pesos=True, cor_destaque="crimson", tamanho_no=650):
    """Desenha o grafo G destacando (em vermelho) a lista de arestas 'destaque'."""
    destacadas = {frozenset((u, v)) for u, v, *_ in (destaque or [])}

    cores    = [cor_destaque if frozenset(e) in destacadas else "lightgray" for e in G.edges]
    larguras = [3.0          if frozenset(e) in destacadas else 1.0         for e in G.edges]

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=cores, width=larguras)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#1D3557", node_size=tamanho_no)
    nx.draw_networkx_labels(G, pos, ax=ax, font_color="white",
                            font_weight="bold", font_size=9)
    if mostrar_pesos:
        nx.draw_networkx_edge_labels(G, pos, ax=ax, font_size=8,
                                     edge_labels=nx.get_edge_attributes(G, peso))
    ax.set_title(titulo)
    ax.axis("off")


fig, eixo = plt.subplots(figsize=(7, 5))
desenhar_rede(G, pos, eixo, "Instância didática: qual o subconjunto de arestas\n"
                            "de menor custo que mantém tudo conectado?")
plt.show()

### 4.1 `minimum_spanning_tree`

In [ ]:
T = nx.minimum_spanning_tree(G, weight="weight", algorithm="kruskal")

print("Tipo do objeto devolvido:", type(T).__name__)
print("Arestas da AGM   :", sorted(T.edges))
print("É uma árvore?    :", nx.is_tree(T))
print("Número de arestas:", T.number_of_edges(), "( = |V| - 1 =", G.number_of_nodes() - 1, ")")
print("Cobre todos os vértices?", set(T.nodes) == set(G.nodes))
print(f"Custo total      : {T.size(weight='weight'):.0f}")

> **Atenção ao cálculo do custo:** `T.size()` conta arestas, enquanto
> `T.size(weight="weight")` **soma os pesos**. É a forma mais curta e segura de
> obter o custo da solução.

In [ ]:
# Os dois algoritmos devem chegar ao mesmo custo:
for algoritmo in ["kruskal", "prim"]:
    T_alg = nx.minimum_spanning_tree(G, weight="weight", algorithm=algoritmo)
    print(f"{algoritmo:<8} | custo = {T_alg.size(weight='weight'):>5.0f} | "
          f"arestas = {sorted(T_alg.edges)}")

### 4.2 `minimum_spanning_edges`

Útil quando queremos apenas a **lista** de arestas (por exemplo, para montar um
relatório) sem construir um novo grafo.

In [ ]:
selecionadas = list(nx.minimum_spanning_edges(G, weight="weight", data=True))

pd.DataFrame([{"de": u, "para": v, "custo": d["weight"]} for u, v, d in selecionadas]) \
  .sort_values("custo") \
  .reset_index(drop=True)

### 4.3 Visualizando a solução

In [ ]:
fig, (e1, e2) = plt.subplots(1, 2, figsize=(13, 5))

desenhar_rede(G, pos, e1, f"Rede original — custo de construir tudo: {G.size(weight='weight'):.0f}")
desenhar_rede(G, pos, e2, f"Árvore Geradora Mínima — custo: {T.size(weight='weight'):.0f}",
              destaque=T.edges)

plt.tight_layout()
plt.show()

Repare que a aresta `A - B` (custo 4) **não** foi escolhida: `A` e `B` já ficam
ligados, mais barato, através de `C` (arestas `B-C` de custo 1 e `A-C` de custo 2).
Essa é exatamente a decisão que o algoritmo de Kruskal toma quando rejeita uma
aresta por "formar ciclo".

---
---
## 5. Uma instância maior: backbone nacional de fibra óptica
---
---

Uma operadora de telecomunicações vai construir um **backbone** de fibra óptica
interligando as **21 capitais** brasileiras listadas abaixo. Por questões de
licenciamento e de custo, o cabo só pode ser lançado ao longo dos principais eixos
rodoviários já existentes — são esses os **trechos possíveis**, com suas distâncias
rodoviárias aproximadas em quilômetros.

Todas as capitais precisam ficar conectadas à rede, direta ou indiretamente.
**Quais trechos devem ser efetivamente construídos** para que a extensão total de
fibra seja a menor possível?

Este é um problema de AGM: queremos um subconjunto de trechos que (i) conecte
todas as 21 capitais e (ii) tenha comprimento total mínimo.

### 5.1 Modelagem

Além do nome, vamos guardar em cada vértice a sua **coordenada geográfica**
(longitude, latitude). Isso não entra no cálculo, mas permitirá desenhar a rede
sobre o mapa do Brasil em vez de usar um layout automático.

Desta vez usaremos um atributo de nome próprio, `distancia`, em vez do `weight`
padrão — e por isso precisaremos informá-lo em todas as chamadas.

In [ ]:
# vértice: (nome, longitude, latitude)
capitais = {
     1: ("Belém",          -48.50,  -1.46),   2: ("São Luís",       -44.30,  -2.53),
     3: ("Teresina",       -42.80,  -5.09),   4: ("Fortaleza",      -38.54,  -3.73),
     5: ("Natal",          -35.21,  -5.79),   6: ("João Pessoa",    -34.86,  -7.12),
     7: ("Recife",         -34.88,  -8.05),   8: ("Maceió",         -35.74,  -9.65),
     9: ("Aracaju",        -37.07, -10.91),  10: ("Salvador",       -38.50, -12.97),
    11: ("Brasília",       -47.88, -15.79),  12: ("Goiânia",        -49.25, -16.68),
    13: ("Cuiabá",         -56.10, -15.60),  14: ("Campo Grande",   -54.62, -20.44),
    15: ("Belo Horizonte", -43.94, -19.92),  16: ("Vitória",        -40.34, -20.32),
    17: ("Rio de Janeiro", -43.20, -22.91),  18: ("São Paulo",      -46.63, -23.55),
    19: ("Curitiba",       -49.27, -25.43),  20: ("Florianópolis",  -48.55, -27.59),
    21: ("Porto Alegre",   -51.23, -30.03),
}

# (capital A, capital B, distância rodoviária aproximada em km)
trechos = [
    ( 1,  2,  806), ( 1,  3,  921), ( 1, 11, 2119), ( 2,  3,  446), ( 3,  4,  634),
    ( 3,  7, 1136), ( 3, 11, 1789), ( 4,  5,  537), ( 4,  7,  800), ( 5,  6,  185),
    ( 6,  7,  120), ( 7,  8,  285), ( 7, 10,  839), ( 8,  9,  294), ( 9, 10,  356),
    (10, 11, 1446), (10, 15, 1372), (10, 16, 1201), (11, 12,  209), (11, 13, 1133),
    (11, 15,  716), (12, 13,  934), (12, 14,  848), (12, 15,  906), (13, 14,  694),
    (14, 18, 1014), (14, 19,  991), (15, 16,  524), (15, 17,  434), (15, 18,  586),
    (16, 17,  521), (17, 18,  430), (18, 19,  408), (18, 21, 1109), (19, 20,  300),
    (20, 21,  476),
]

Brasil = nx.Graph()
Brasil.add_nodes_from((i, {"cidade": nome, "lon": lon, "lat": lat})
                      for i, (nome, lon, lat) in capitais.items())
Brasil.add_edges_from((u, v, {"distancia": d}) for u, v, d in trechos)

nomes = {i: capitais[i][0] for i in Brasil.nodes}

print(f"|V| = {Brasil.number_of_nodes()} capitais")
print(f"|A| = {Brasil.number_of_edges()} trechos possíveis")
print(f"A rede é conexa? {nx.is_connected(Brasil)}")
print(f"Extensão total se construíssemos TODOS os trechos: "
      f"{Brasil.size(weight='distancia'):.0f} km")

### 5.2 Resolvendo

In [ ]:
AGM = nx.minimum_spanning_tree(Brasil, weight="distancia", algorithm="prim")

print(f"Trechos a construir : {AGM.number_of_edges()}  "
      f"(de {Brasil.number_of_edges()} possíveis)")
print(f"Extensão total      : {AGM.size(weight='distancia'):.0f} km")
print(f"Economia em relação a construir tudo: "
      f"{Brasil.size(weight='distancia') - AGM.size(weight='distancia'):.0f} km "
      f"({100 * (1 - AGM.size(weight='distancia') / Brasil.size(weight='distancia')):.1f}%)")
print(f"\nA solução é mesmo uma árvore? {nx.is_tree(AGM)}")
print(f"Ela cobre todas as capitais?   {set(AGM.nodes) == set(Brasil.nodes)}")

Uma tabela com os nomes das cidades comunica muito melhor o resultado do que
uma lista de números.

In [ ]:
plano = pd.DataFrame([
    {"de": nomes[u], "para": nomes[v], "km": d["distancia"]}
    for u, v, d in AGM.edges(data=True)
]).sort_values("km", ascending=False).reset_index(drop=True)

plano.index += 1
print(f"Plano de instalação — {len(plano)} trechos, {plano['km'].sum()} km no total\n")
plano

### 5.3 Visualizando a solução

Como guardamos a longitude e a latitude de cada capital, podemos usá-las
diretamente como posição dos vértices: o desenho fica parecido com um mapa.

Usamos o **mesmo** dicionário de posições nos dois painéis para permitir a
comparação visual. Como a rede tem 21 vértices, omitimos os rótulos das arestas.

In [ ]:
# posição = coordenada geográfica real de cada capital
pos_brasil = {i: (dados["lon"], dados["lat"]) for i, dados in Brasil.nodes(data=True)}

# rótulos ligeiramente acima do vértice, para não cobrirem o marcador
pos_rotulos = {i: (lon, lat + 0.9) for i, (lon, lat) in pos_brasil.items()}

fig, (e1, e2) = plt.subplots(1, 2, figsize=(16, 8))

for eixo, (titulo, destaque) in zip(
        [e1, e2],
        [(f"Trechos POSSÍVEIS — {Brasil.number_of_edges()} trechos, "
          f"{Brasil.size(weight='distancia'):.0f} km", None),
         (f"Árvore Geradora Mínima — {AGM.number_of_edges()} trechos, "
          f"{AGM.size(weight='distancia'):.0f} km", AGM.edges)]):

    desenhar_rede(Brasil, pos_brasil, eixo, titulo, destaque=destaque,
                  peso="distancia", mostrar_pesos=False, tamanho_no=120)
    nx.draw_networkx_labels(Brasil, pos_rotulos, ax=eixo, labels=nomes,
                            font_size=7, font_color="black",
                            verticalalignment="bottom",
                            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                      ec="none", alpha=0.75))
    eixo.set_aspect("equal")

plt.tight_layout()
plt.show()

---
## 6. Variações do problema
---

Pequenas mudanças no enunciado aparecem o tempo todo na prática. Todas podem ser
resolvidas com as mesmas funções do NetworkX.

### 6.1 Árvore geradora **máxima**

Às vezes queremos **maximizar** o valor das arestas escolhidas. Exemplo clássico:
cada ligação tem uma **confiabilidade** e queremos a rede conectada mais confiável.

`nx.maximum_spanning_tree` resolve diretamente.

In [ ]:
T_max = nx.maximum_spanning_tree(G, weight="weight")

print(f"MÍNIMA : arestas {sorted(T.edges)} → custo {T.size(weight='weight'):.0f}")
print(f"MÁXIMA : arestas {sorted(T_max.edges)} → custo {T_max.size(weight='weight'):.0f}")

fig, (e1, e2) = plt.subplots(1, 2, figsize=(13, 5))
desenhar_rede(G, pos, e1, f"Árvore geradora MÍNIMA — custo {T.size(weight='weight'):.0f}",
              destaque=T.edges)
desenhar_rede(G, pos, e2, f"Árvore geradora MÁXIMA — custo {T_max.size(weight='weight'):.0f}",
              destaque=T_max.edges, cor_destaque="darkgreen")
plt.tight_layout()
plt.show()

### 6.2 E se o grafo for **desconexo**?

Se o grafo não é conexo, **não existe** árvore geradora — não há como ligar tudo
usando apenas as arestas disponíveis. Nesse caso, `nx.minimum_spanning_tree`
devolve a **floresta geradora mínima**: a AGM de cada componente.

Sempre vale a pena checar `nx.is_connected` **antes** de resolver.

In [ ]:
H = G.copy()
H.remove_edges_from([("B", "D"), ("C", "D"), ("C", "E")])   # separa {A,B,C} de {D,E,F}

F = nx.minimum_spanning_tree(H, weight="weight")

print("H é conexo?", nx.is_connected(H))
print("Componentes:", [sorted(c) for c in nx.connected_components(H)])
print()
print("O resultado é uma árvore?", nx.is_tree(F), "| É uma floresta?", nx.is_forest(F))
print(f"Arestas: {F.number_of_edges()} "
      f"( = |V| - nº de componentes = {H.number_of_nodes()} - "
      f"{nx.number_connected_components(H)} )")
print(f"Custo total: {F.size(weight='weight'):.0f}")

fig, eixo = plt.subplots(figsize=(7, 5))
desenhar_rede(H, pos, eixo, "Floresta geradora mínima (grafo desconexo)", destaque=F.edges)
plt.show()

### 6.3 Aresta **proibida**

*"O trecho X–Y não pode ser construído"* (licença ambiental negada, terreno
indisponível). Basta **removê-la** do grafo e resolver normalmente — sem esquecer
de verificar se o grafo continua conexo.

In [ ]:
G_proibida = G.copy()
G_proibida.remove_edge("B", "C")

print("Sem a aresta B-C o grafo continua conexo?", nx.is_connected(G_proibida))

T_proibida = nx.minimum_spanning_tree(G_proibida, weight="weight")

print(f"Solução : {sorted(T_proibida.edges)} → custo {T_proibida.size(weight='weight'):.0f}")
print(f"Custo da restrição 'não usar B-C': "
      f"{T_proibida.size(weight='weight') - T.size(weight='weight'):.0f}")

### 6.4 Aresta **obrigatória**

*"O trecho X–Y tem que ser construído"* (contrato já assinado, exigência técnica).

O truque é atribuir a ela um custo **menor que o de todas as outras**. Como o
algoritmo guloso analisa as arestas em ordem crescente, ela será a primeira
escolhida e, por não fechar ciclo nenhum, entrará com certeza na solução. Depois,
recuperamos o subgrafo correspondente **no grafo original** para calcular o custo
verdadeiro.

In [ ]:
G_forcada = G.copy()
G_forcada["C"]["E"]["weight"] = min(nx.get_edge_attributes(G, "weight").values()) - 1

T_auxiliar = nx.minimum_spanning_tree(G_forcada, weight="weight")

# subgrafo do grafo ORIGINAL com as arestas escolhidas (pesos verdadeiros)
T_forcada = G.edge_subgraph(T_auxiliar.edges).copy()

print("A aresta C-E entrou na solução?", T_forcada.has_edge("C", "E"))
print(f"Solução : {sorted(T_forcada.edges)} → custo {T_forcada.size(weight='weight'):.0f}")
print(f"Custo da exigência 'construir C-E': "
      f"{T_forcada.size(weight='weight') - T.size(weight='weight'):.0f}")

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(18, 5))

for eixo, (titulo, arvore, cor) in zip(eixos, [
        (f"Sem restrições — custo {T.size(weight='weight'):.0f}",          T,          "crimson"),
        (f"B-C proibida — custo {T_proibida.size(weight='weight'):.0f}",   T_proibida, "purple"),
        (f"C-E obrigatória — custo {T_forcada.size(weight='weight'):.0f}", T_forcada,  "darkorange")]):
    desenhar_rede(G, pos, eixo, titulo, destaque=arvore.edges, cor_destaque=cor)

plt.tight_layout()
plt.show()

---
## 7. Por que um algoritmo guloso funciona?
---

Algoritmos gulosos raramente produzem a solução ótima. A AGM é uma exceção
notável, e a razão é a **propriedade do corte**:

> **Propriedade do corte.** Seja $S \subset V$ um subconjunto próprio e não vazio
> de vértices. Considere o **corte** formado por todas as arestas com uma ponta em
> $S$ e outra em $V \setminus S$. Se uma aresta $e$ é a **de menor custo** desse
> corte (e é única nesse mínimo), então $e$ pertence a **toda** árvore geradora mínima.

*Esboço da justificativa:* suponha uma árvore geradora $T$ que não contenha $e$.
Como $T$ conecta todos os vértices, ela precisa cruzar o corte por alguma outra
aresta $f$. Trocando $f$ por $e$, continuamos com uma árvore geradora, agora de
custo estritamente menor — logo $T$ não era mínima.

É exatamente isso que os algoritmos exploram:

- **Prim** define o corte como (vértices já visitados) × (não visitados) e escolhe
  a aresta mais barata desse corte;
- **Kruskal** aceita a aresta mais barata que liga duas componentes distintas — o
  corte é dado pela componente de um dos extremos.

Uma consequência prática: se **todos os custos forem distintos**, a AGM é
**única**. Havendo empates, *pode* haver mais de uma AGM — todas com o mesmo custo
total (mas o empate não garante que haja).

Como nossa instância é pequena, podemos verificar isso por **enumeração**.

In [ ]:
custos = list(nx.get_edge_attributes(G, "weight").values())
print("Custos das arestas:", sorted(custos))
print("Todos distintos? ", len(custos) == len(set(custos)),
      "→ há um empate (o custo 2 aparece duas vezes), logo NÃO temos garantia de unicidade.")

custo_otimo = T.size(weight="weight")
otimas = [A for A in nx.SpanningTreeIterator(G) if A.size(weight="weight") == custo_otimo]

print(f"\nTotal de árvores geradoras : {round(nx.number_of_spanning_trees(G))}")
print(f"Custo mínimo               : {custo_otimo:.0f}")
print(f"Árvores de custo mínimo    : {len(otimas)}")
print("\nNeste caso a AGM acabou sendo única mesmo com o empate: as duas arestas de")
print("custo 2 (A-C e D-E) estão em cortes diferentes, e por isso não competem entre si.")

---
---
# 8. EXERCÍCIOS
---
---

Todos os dados necessários estão nas células de código. Execute a célula de dados
e programe a resposta logo abaixo.

### Exercício 1 — Rede de fibra óptica no campus

A Superintendência de Tecnologia da Informação quer interligar por fibra óptica
nove pontos do campus. A lista abaixo traz os trechos onde é **possível** passar
o cabo e a metragem de cada um. A obra deve conectar todos os nove pontos com o
menor comprimento total de cabo.

Pede-se:

**(a)** Construa o grafo ponderado e verifique se ele é conexo.
**(b)** Quantos trechos terá, necessariamente, qualquer solução viável? Por quê?
**(c)** Encontre a AGM com o algoritmo de **Kruskal** e informe a metragem total.
**(d)** Refaça com o algoritmo de **Prim** e confirme que o custo é o mesmo.
**(e)** Monte um `DataFrame` com o plano de obra (trechos escolhidos, ordenados por metragem).
**(f)** Qual o percentual de economia em relação a construir **todos** os trechos possíveis?
**(g)** Desenhe, lado a lado, a rede de trechos possíveis e a solução encontrada.

In [ ]:
campus = [
    ("Reitoria",     "Biblioteca",   120), ("Reitoria",     "Bloco A",      200),
    ("Biblioteca",   "Bloco A",       90), ("Biblioteca",   "Bloco B",      150),
    ("Bloco A",      "Bloco B",      110), ("Bloco A",      "Laboratórios", 260),
    ("Bloco B",      "Laboratórios", 140), ("Bloco B",      "Restaurante",  180),
    ("Laboratórios", "Restaurante",   95), ("Laboratórios", "Data Center",  130),
    ("Restaurante",  "Quadra",       210), ("Data Center",  "Quadra",       160),
    ("Data Center",  "Almoxarifado", 175), ("Quadra",       "Almoxarifado",  80),
]

# Seu código a partir daqui:


### Exercício 2 — Rede de irrigação com restrições de projeto

Uma fazenda vai construir uma rede de tubulação ligando a captação de água aos
seus setores produtivos. Os custos abaixo estão em milhares de reais.

Pede-se:

**(a)** Determine a rede de custo mínimo, **sem restrições**, e informe o custo.
**(b)** Por exigência técnica, o trecho `("Captação", "Reservatório 2")` **precisa** ser construído. Qual passa a ser a solução e o custo? Quanto custou essa exigência?
**(c)** Uma questão fundiária impede a construção do trecho `("Setor Sul", "Setor Oeste")`. Refaça o item (a) sem esse trecho. Qual o custo dessa restrição?
**(d)** Aplique as duas restrições **simultaneamente** e apresente a solução final.
**(e)** Desenhe as quatro soluções em uma figura com quatro painéis, usando **as mesmas posições** dos vértices.
**(f)** Ordene as quatro soluções por custo e comente: qual restrição foi mais cara para a fazenda?

> *Dica:* reproduza as ideias das Seções 6.3 e 6.4 — remover a aresta proibida e
> atribuir custo fictício à obrigatória.

In [ ]:
irrigacao = [
    ("Captação",       "Reservatório 1", 40), ("Captação",       "Reservatório 2", 55),
    ("Reservatório 1", "Reservatório 2", 30), ("Reservatório 1", "Setor Norte",    60),
    ("Reservatório 2", "Setor Norte",    45), ("Reservatório 2", "Setor Sul",      70),
    ("Setor Norte",    "Setor Leste",    35), ("Setor Sul",      "Setor Leste",    50),
    ("Setor Sul",      "Setor Oeste",    25), ("Setor Leste",    "Setor Oeste",    65),
]

# Seu código a partir daqui:


### Exercício 3 — Rede de comunicação mais confiável

Seis estações de uma rede de comunicação podem ser interligadas pelos enlaces
abaixo. O número associado a cada enlace é a sua **confiabilidade**: a
probabilidade de o enlace operar sem falha.

Pede-se:

**(a)** Construa o grafo com o atributo `confiabilidade`.
**(b)** Encontre a árvore geradora que **maximiza a soma** das confiabilidades. Quais enlaces são escolhidos?
**(c)** Compare com a árvore geradora **mínima**. Comente por que a mínima seria uma péssima escolha aqui.
**(d)** Para a solução do item (b), calcule o **produto** das confiabilidades — a probabilidade de que **todos** os enlaces escolhidos estejam operando simultaneamente (supondo independência).
**(e) (desafio)** Maximizar a **soma** e maximizar o **produto** não são, em princípio, o mesmo problema. Mostre que maximizar $\prod_e p_e$ equivale a minimizar $\sum_e \left(-\ln p_e\right)$ e resolva o problema correto: crie um atributo `custo_log` com $-\ln p_e$ e aplique `nx.minimum_spanning_tree`. A solução coincide com a do item (b)? Explique o porquê do resultado obtido.

In [ ]:
import math

enlaces = [
    ("E1", "E2", 0.95), ("E1", "E3", 0.90), ("E1", "E4", 0.80),
    ("E2", "E3", 0.99), ("E2", "E4", 0.85), ("E3", "E5", 0.92),
    ("E4", "E5", 0.97), ("E4", "E6", 0.88), ("E5", "E6", 0.93),
]

# Seu código a partir daqui:


### Exercício 4 — Monitoramento ambiental: rede fragmentada

Uma rede de estações de monitoramento de qualidade do ar teve várias ligações
interrompidas. A lista abaixo traz as ligações que **ainda podem** ser ativadas,
com o custo mensal de operação de cada uma.

Pede-se:

**(a)** Construa o grafo (não esqueça a estação isolada). Ele é conexo? Quantas componentes existem?
**(b)** Explique por que **não existe** árvore geradora nesse grafo.
**(c)** Obtenha a **floresta geradora mínima** e informe o custo total.
**(d)** Confirme numericamente a relação: nº de arestas da floresta $= |V| -$ nº de componentes.
**(e)** Apresente, em um `DataFrame`, o custo da AGM de **cada componente** separadamente. *Dica:* itere sobre `nx.connected_components` e use `G.subgraph(componente)`.
**(f)** Desenhe a rede pintando cada componente de uma cor e destacando as ligações escolhidas.
**(g)** Suponha que seja possível criar **novas** ligações entre estações de componentes diferentes, ao custo de 500 cada. Qual o custo total de tornar toda a rede conexa? Quantas novas ligações são necessárias?

In [ ]:
monitoramento = [
    ("M01", "M02", 120), ("M02", "M03",  95), ("M01", "M03", 140), ("M03", "M04", 110),
    ("M04", "M05", 130), ("M02", "M05", 175),
    ("M06", "M07",  85), ("M07", "M08", 100), ("M06", "M08", 155), ("M08", "M09",  90),
    ("M10", "M11", 145), ("M11", "M12", 115), ("M10", "M12", 160),
]
estacoes_isoladas = ["M13"]

# Seu código a partir daqui:


### Exercício 5 — Comparando os algoritmos em uma rede grande

Gere a rede aleatória ponderada abaixo e responda:

**(a)** Qual a ordem, o tamanho e a densidade da rede? Ela é conexa?
**(b)** Resolva a AGM com `kruskal` e com `prim`. Os **custos** coincidem? E os **conjuntos de arestas**?
**(c)** Verifique, para cada solução, que o resultado é de fato uma árvore (`nx.is_tree`) com $|V|-1$ arestas.
**(d)** Meça o tempo de execução dos dois algoritmos com `%timeit`. Comente o resultado à luz da tabela comparativa da Seção 3.1.
**(e)** Repita os itens (b) e (d) para uma rede **densa**: refaça a geração com `p = 0.7`. Muda alguma coisa na comparação de tempos?
**(f)** Qual o percentual das arestas da rede que entra na AGM? Como esse percentual varia entre a rede esparsa e a densa? Interprete.

In [ ]:
import random

R = nx.gnp_random_graph(200, 0.05, seed=2024)

sorteio = random.Random(2024)
for u, v in R.edges:
    R[u][v]["weight"] = sorteio.randint(1, 100)

# Seu código a partir daqui:


### Exercício 6 — Desafio: análise de sensibilidade e a segunda melhor árvore

Trabalhe com a instância didática `G` da Seção 4 (ou com a rede do Exercício 1).

**(a)** Para cada aresta que **está** na AGM, determine em quanto seu custo poderia
**aumentar** antes que ela deixasse de ser escolhida. *Dica:* remova a aresta,
recalcule a AGM e veja qual aresta a substituiu.

**(b)** Para cada aresta que **não está** na AGM, determine em quanto seu custo
precisaria **diminuir** para que ela passasse a fazer parte da solução.

**(c)** Encontre a **segunda melhor** árvore geradora, isto é, a de menor custo
entre todas as que são diferentes da AGM. *Dica:* para cada aresta $e$ da AGM,
calcule a AGM de $G - e$; a melhor dessas soluções é a resposta.

**(d)** Confirme sua resposta do item (c) por **enumeração**, usando
`nx.SpanningTreeIterator` (só é viável porque o grafo é pequeno!). Qual a
diferença de custo entre a melhor e a segunda melhor?

**(e)** Interprete os resultados de (a) e (b) do ponto de vista gerencial: quais
trechos do projeto são "robustos" a variações de preço e quais estão "no limite"
de entrar ou sair da solução?

In [ ]:
# Seu código a partir daqui:


---
---

## Para a próxima aula

Na **Aula 04** trataremos do **Problema do Caminho Mais Curto**. Vale antecipar a
diferença conceitual: a AGM minimiza o custo de **construir** a rede inteira; o
caminho mais curto minimiza o custo de **percorrer** a rede entre dois pontos
específicos. Não confunda os dois — a AGM raramente contém o caminho mais curto
entre um par qualquer de vértices.

### Documentação

- Algoritmos de árvore no NetworkX: <https://networkx.org/documentation/stable/reference/algorithms/tree.html>
- `nx.minimum_spanning_tree`: <https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.tree.mst.minimum_spanning_tree.html>
- `nx.SpanningTreeIterator`: <https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.tree.mst.SpanningTreeIterator.html>